# 16 · Callbacks and custom pipelines

There are two ways to add code of your own to an experiment. A **callback**
keeps the configuration-driven pipelines and runs your code at fixed points
of them; a **pipeline of your own** calls WarpRec's components from Python
and decides everything itself. This guide covers:

- a `WarpRecCallback` subclass in `user_code/`, configured under
  `general.callback`: `on_data_reading` changes the frame just read,
  `on_dataset_creation` fills the stash of every dataset, `on_training_complete`
  and `on_evaluation_complete` see each model, and `on_trial_result`, a Ray
  Tune hook, records every validation report;
- where each hook runs: all of them in your process for the design pipeline,
  but split between a Ray worker and your process for the train pipeline, and
  what that means for state the callback keeps;
- a custom metric that reads what the callback put in the stash, end to end;
- a pipeline written against the Python API with no configuration at all:
  read, filter, split, build the datasets, select hyperparameters on
  validation, train an iterative model with Lightning as the design pipeline
  does, test, run a significance test and write the results.

The [Callbacks](https://warprec.readthedocs.io/en/latest/extending/callbacks/)
and [Stash](https://warprec.readthedocs.io/en/latest/extending/stash/) pages
describe the same interfaces in prose.

**Requirements:** `pip install warprec jupyter`. Runs in three to five minutes
on a laptop CPU, most of it Ray starting processes for the train pipeline. It reads
MovieLens-100K through `guide_data.movielens_100k()`, which downloads and
converts it exactly as guide 1 shows.

In [1]:
import contextlib
import inspect
import io
import json
import logging
import os
import re
import shutil
import sys
import warnings
from pathlib import Path

# Ray prints INFO lines (cluster start, package upload) through handlers bound
# to the notebook when it is imported, which no redirection catches; keep only
# its warnings.
os.environ["RAY_LOGGER_LEVEL"] = "warning"

import lightning as L
import pandas as pd
import ray
import torch
from IPython.utils.capture import capture_output
from loguru import logger

sys.path.insert(0, "..")
from guide_data import movielens_100k

folder = movielens_100k()
RUNS = Path("../runs/16-callbacks-and-custom-pipelines")
shutil.rmtree(RUNS, ignore_errors=True)
RUNS.mkdir(parents=True)

pd.set_option("display.width", 120)

# Lightning reports the hardware it found and warns about loader workers; none
# of it is about WarpRec, so it is kept out of the outputs.
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", module="lightning")
warnings.filterwarnings("ignore", module="torch")
logger.disable("warprec")


def show(obj):
    """Print the source of a class or function without its docstrings."""
    source = inspect.getsource(obj)
    print(re.sub(r'\n\s*""".*?"""', "", source, flags=re.S))

## The callback and its configuration

A callback is a class that subclasses `WarpRecCallback` and overrides the hooks
it needs; the others do nothing. This guide's callback is
`GuideCallback` in `user_code/my_callback.py`, next to the metric of the
third section:

In [2]:
sorted(p.name for p in Path("user_code").glob("*.py"))

['my_callback.py', 'my_metrics.py']

A configuration names it in the `general.callback` section, which every
pipeline reads:

In [3]:
print(Path("configs/design.yml").read_text())

# The design pipeline with the guide's callback and its stash-reading metric.
# Paths are relative to the guide's folder.
general:
  device: cpu
  # The metric is imported like any custom module; the callback is not listed
  # here, it has a section of its own.
  custom_modules:
    - user_code/my_metrics.py
  callback:
    callback_path: user_code/my_callback.py
    callback_name: GuideCallback
    # Passed to the constructor as GuideCallback(*args, **kwargs), unvalidated.
    kwargs:
      items_path: ../data/ml-100k/items.tsv
      output_dir: ../runs/16-callbacks-and-custom-pipelines/design
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
models:
  EASE:
    l2: 200.0
    optimization:
      num_workers: 0
  BPR:
    embedding_size: 64
    reg_weight: 0.0001
    batch_size: 1024
    epochs: 5
    learning_rat

- `callback_path` is the `.py` file, relative to the directory WarpRec is
  started from; `callback_name` the class in it. Loading the configuration
  executes the file and checks that the class exists and subclasses
  `WarpRecCallback`, so a typo stops the run before any data is read.
- `args` and `kwargs` are passed to the constructor as they are,
  `GuideCallback(*args, **kwargs)`. WarpRec does not validate them.
- The callback file is not a `custom_modules` entry, and does not need to be:
  it is executed from its path, not imported by name. The metric it feeds is a
  custom module, imported as guide 15 shows.

The pipelines build the callback with `load_callback`, the same call used
here. `WarpRecCallback` is a Ray Tune `Callback` too, which is why the class
also has Tune's hooks:

In [4]:
from warprec.utils.config import load_callback, load_design_configuration

config = load_design_configuration("configs/design.yml")
section = config.general.callback
callback = load_callback(section, *section.args, **section.kwargs)

print(
    type(callback).__name__, "is a", [c.__name__ for c in type(callback).__mro__[1:3]]
)
print("in sys.modules:", type(callback).__module__ in sys.modules)

GuideCallback is a ['WarpRecCallback', 'Callback']
in sys.modules: False


The module was executed without being registered in `sys.modules`. That is
what lets the train pipeline send the callback to Ray workers that do not
have the file: Ray's serializer sees a class it cannot import, and ships the
class itself along with the object.

## The hooks

The constructor stores the paths it is given, already resolved, and three
small containers. Every hook also appends a line to `events.jsonl` in
`output_dir` with its name and the ID of the process it ran in, through
`_event`; `__setstate__` does the same each time a copy of the callback is
rebuilt from a pickle. The rest of the guide reads that file to see where the
pipelines call what.

In [5]:
sys.path.insert(0, "user_code")
import my_callback

show(my_callback.GuideCallback.__init__)
show(my_callback.GuideCallback.__setstate__)

    def __init__(
        self,
        *args: Any,
        items_path: str = "items.tsv",
        output_dir: str = ".",
        metric: str = "nDCG@10",
        **kwargs: Any,
    ):
        super().__init__(*args, **kwargs)
        self.metric = metric
        # Resolved here, in the process that reads the configuration, so that a
        # Ray worker started in another directory reads the same files.
        self.items_path = Path(items_path).resolve()
        self.output_dir = Path(output_dir).resolve()
        self.output_dir.mkdir(parents=True, exist_ok=True)

        self.rows_kept: Optional[int] = None
        self.trial_reports: List[Dict[str, Any]] = []
        self.test_results: List[Dict[str, Any]] = []
        self._event("__init__")

    def __setstate__(self, state: Dict[str, Any]):
        self.__dict__.update(state)
        self._event("unpickled")



### `on_data_reading`

Called with the frame the reader returned, before the `filtering` section and
the split; whatever it returns replaces the frame. It is called only when the
data is read as one file (`loading_strategy: dataset`); data loaded already
split skips it.

The frame is a Narwhals frame, as guide 1 shows. `GuideCallback` drops the
ratings dated before the film's release date, a check that needs the items
file, which no filter reads. A transform a filter can express (a rating
threshold, a k-core) belongs in `filtering` (guide 2).

In [6]:
show(my_callback.GuideCallback.on_data_reading)

    def on_data_reading(self, data):
        dates = self._release_dates().dropna()
        released = nw.from_dict(
            {
                "item_id": dates.item_id.tolist(),
                "released": (dates.released.astype("int64") // 10**9).tolist(),
            },
            backend=nw.get_native_namespace(data),
        ).with_columns(nw.col("item_id").cast(data.schema["item_id"]))

        kept = (
            data.join(released, on="item_id", how="left")
            .filter(
                nw.col("released").is_null()
                | (nw.col("timestamp") >= nw.col("released"))
            )
            .drop("released")
        )
        self.rows_kept = len(kept)
        self._event("on_data_reading", rows_read=len(data), rows_kept=len(kept))
        return kept



### `on_dataset_creation`

Called once every dataset of the run exists: the main dataset (train/test),
the validation dataset (train/validation) or `None`, and the list of
cross-validation folds, empty without them. It is the place to fill the
**stash** (guide 4), because it sees every dataset. `GuideCallback` stores
each film's release year under `item_year`, built through each dataset's own
item mapping.

In [7]:
show(my_callback.GuideCallback.on_dataset_creation)

    def on_dataset_creation(
        self, main_dataset, val_dataset, validation_folds, *args, **kwargs
    ):
        dates = self._release_dates()
        year_by_id = dict(zip(dates.item_id, dates.released.dt.year))

        datasets = [main_dataset, val_dataset, *validation_folds]
        datasets = [dataset for dataset in datasets if dataset is not None]
        for dataset in datasets:
            _, item_map = dataset.get_mappings()
            item_year = torch.full((dataset.get_dims()[1] + 1,), float("nan"))
            for raw_id, index in item_map.items():
                item_year[index] = year_by_id.get(raw_id, float("nan"))
            dataset.add_to_stash("item_year", item_year)

        self._event("on_dataset_creation", datasets=len(datasets))



### `on_training_complete` and `on_evaluation_complete`

Both are called once per model under `models`: the first with the trained
model (a closed-form model is trained when it is built), the second after the
test evaluation, with the model, its parameters and the results. `results` is
what `Evaluator.compute_results()` returns, `{cutoff: {metric: value}}`, with a
tensor of per-user values for per-user metrics (guide 12). `GuideCallback`
averages them and writes a table.

In [8]:
show(my_callback.GuideCallback.on_training_complete)
show(my_callback.GuideCallback.on_evaluation_complete)

    def on_training_complete(self, model, *args, **kwargs):
        self._event(
            "on_training_complete",
            model=model.name,
            rows_kept=self.rows_kept,
            trial_reports=len(self.trial_reports),
        )

    def on_evaluation_complete(self, model, params, results, *args, **kwargs):
        for k, metrics in results.items():
            row = {"model": model.name, "k": k}
            for name, value in metrics.items():
                row[name] = value.nanmean().item() if torch.is_tensor(value) else value
            self.test_results.append(row)

        pd.DataFrame(self.test_results).to_csv(
            self.output_dir / "test_results.tsv", sep="\t", index=False
        )
        if self.trial_reports:
            pd.DataFrame(self.trial_reports).to_json(
                self.output_dir / "trial_reports.jsonl", orient="records", lines=True
            )
        self._event("on_evaluation_complete", model=model.name)



### A Ray Tune hook

The train pipeline gives the callback to Ray Tune, so every hook of
[`ray.tune.Callback`](https://docs.ray.io/en/latest/tune/api/doc/ray.tune.Callback.html)
works. `on_trial_result` is called each time a trial reports: once for a
closed-form model, after every evaluation for an iterative one (every
`optimization.eval_every_n` epochs). `result` holds the reported metrics, the
validation metric among them, an iterative model's `train_loss` (the mean
loss of the epoch), and Ray's counters such as `training_iteration`.

In [9]:
show(my_callback.GuideCallback.on_trial_result)

    def on_trial_result(self, iteration, trials, trial, result, **info):
        self.trial_reports.append(
            {
                "trial": str(trial),
                "params": {
                    key: value
                    for key, value in trial.config.items()
                    if not isinstance(value, dict)
                },
                "report": result.get("training_iteration"),
                self.metric: result.get(self.metric),
                "train_loss": result.get("train_loss"),
            }
        )
        self._event("on_trial_result", trial=str(trial))



The design pipeline does not use Ray Tune, so it never calls this hook.

## Running the data hooks from Python

`initialize_datasets` is where every pipeline reads, filters and splits the
data, and it calls both data hooks. With the callback loaded above:

In [10]:
from warprec.common.initialize import initialize_datasets
from warprec.data.reader import ReaderFactory

dataset, val_dataset, folds = initialize_datasets(
    ReaderFactory.get_reader(config), callback, config
)
print("rows kept by on_data_reading:", callback.rows_kept)
print("validation dataset:", val_dataset, "| folds:", folds)
{key: tuple(value.shape) for key, value in dataset.get_stash().items()}

rows kept by on_data_reading: 99769
validation dataset: None | folds: []


{'item_year': (1613,)}

`on_data_reading` dropped 231 of the 100,000 ratings, and there is no
validation split in this configuration, so the callback filled one stash. The
entry has one slot per item plus the padding slot. Through the
dataset's mapping, the years land at each film's index; the film without a
release date gets NaN:

In [11]:
items = pd.read_csv(folder / "items.tsv", sep="\t").set_index("item_id")
_, item_map = dataset.get_mappings()
item_year = dataset.get_stash()["item_year"]
pd.DataFrame(
    {
        "title": items.title[[1, 50, 267]].values,
        "index": [item_map[i] for i in (1, 50, 267)],
        "item_year": [item_year[item_map[i]].item() for i in (1, 50, 267)],
    }
)

,title,index,item_year
0,Toy Story (1995),0,1995.0
1,Star Wars (1977),49,1977.0
2,unknown,266,NaN


Every hook so far ran in this process:

In [12]:
def events(output_dir: Path) -> pd.DataFrame:
    """The callback's events.jsonl, with the process of each line named."""
    lines = (output_dir / "events.jsonl").read_text().splitlines()
    frame = pd.DataFrame([json.loads(line) for line in lines])
    data_pid = frame.pid[frame.hook == "on_data_reading"].iloc[0]
    frame["process"] = [
        "this process"
        if pid == os.getpid()
        else "data preparation worker"
        if pid == data_pid
        else "trial process"
        for pid in frame.pid
    ]
    columns = ["hook", "process"] + [
        c for c in frame.columns if c not in ("hook", "process", "pid")
    ]
    return frame[columns]


events(RUNS / "design")

,hook,process,rows_read,rows_kept,datasets
0,__init__,this process,NaN,NaN,NaN
1,on_data_reading,this process,100000.0,99769.0,NaN
2,on_dataset_creation,this process,NaN,NaN,1.0


## A metric that reads the stash

The evaluator unpacks the dataset's stash into the constructor of every
metric, next to the arguments it passes itself (guide 15). A metric names the
entry it needs as a constructor argument. `NewReleaseShare`, in
`user_code/my_metrics.py`, is the share of each user's top-k released in or
after `since` (1997 by default; the ratings were collected in 1997 and 1998):

In [13]:
import my_metrics

show(my_metrics.NewReleaseShare.__init__)
show(my_metrics.NewReleaseShare.compute_scores)

    def __init__(
        self,
        k: int,
        num_users: int,
        item_year: Tensor,
        *args: Any,
        since: int = 1997,
        dist_sync_on_step: bool = False,
        **kwargs: Any,
    ):
        super().__init__(k, num_users, *args, dist_sync_on_step=dist_sync_on_step)
        self.since = since
        # NaN >= since is False, so an unknown year counts as old.
        self.register_buffer("is_new", (item_year >= since).float())

    def compute_scores(
        self, preds: Tensor, target: Tensor, top_k_rel: Tensor, **kwargs: Any
    ) -> Tensor:
        top_k = self.remap_indices(
            kwargs[f"top_{self.k}_indices"], kwargs.get("item_indices")
        )
        return self.is_new[top_k].mean(dim=1)



`remap_indices` turns positions in the candidate list back into item indices
under sampled evaluation, and does nothing under full evaluation.

The design configuration lists it under `metrics` and imports it through
`custom_modules`; the pipelines build their evaluator with `build_evaluator`,
which passes the stash along. Built by hand, the stash goes in
`additional_data`:

In [14]:
from warprec.evaluation import Evaluator
from warprec.utils.registry import model_registry

ease = model_registry.get(
    "EASE",
    params=config.models["EASE"],
    info=dataset.info(),
    interactions=dataset.train_set,
    **dataset.get_stash(),
)
evaluator = Evaluator(
    list(config.evaluation.metrics),
    list(config.evaluation.top_k),
    train_set=dataset.train_set.get_sparse(),
    additional_data=dataset.get_stash(),
)
evaluator.evaluate(ease, dataset.get_evaluation_dataloader(), "full", dataset)
results = {
    name: value.nanmean().item()
    for name, value in evaluator.compute_results()[10].items()
}
results

{'nDCG': 0.20909172296524048,
 'Recall': 0.12639528512954712,
 'NewReleaseShare': 0.31887590885162354}

31.9% of what EASE puts in a top 10 was released in 1997 or 1998. The same
share by hand, from EASE's top 10 with the training items removed:

In [15]:
train_matrix = torch.from_numpy(dataset.train_set.get_sparse().toarray()).bool()
with torch.inference_mode():
    scores = ease.predict(torch.arange(dataset.get_dims()[0]))
top10 = scores.masked_fill(train_matrix, -torch.inf).topk(10, dim=1).indices
print(f"by hand {(item_year[top10] >= 1997).float().mean().item():.6f}")

by hand 0.318876


A dataset whose stash lacks the entry cannot build the metric at all. This is
why `on_dataset_creation` fills the validation dataset and the folds too: in
the train pipeline every trial builds an evaluator with all of `metrics` from
the dataset it is validated on.

In [16]:
try:
    Evaluator(["NewReleaseShare"], [10], train_set=dataset.train_set.get_sparse())
except TypeError as error:
    print(error)

NewReleaseShare.__init__() missing 1 required positional argument: 'item_year'


## The design pipeline

The design pipeline runs everything in the process that calls it: it loads
the callback, reads the data, builds and trains the two models, and tests
them. Its console output (Lightning's progress bar among it) goes to a log
file here; what matters is what the callback recorded.

In [17]:
from warprec.pipelines import design_pipeline, train_pipeline

logger.remove()  # WarpRec's console output; each run logs to a file instead
logger.enable("warprec")


def run(pipeline, config_path: str, output_dir: Path) -> None:
    """Run a pipeline from a clean output folder, logging to a file there."""
    shutil.rmtree(output_dir, ignore_errors=True)
    output_dir.mkdir(parents=True)
    with (
        open(output_dir / "pipeline.log", "w") as log,
        contextlib.redirect_stdout(log),
        contextlib.redirect_stderr(log),
        capture_output(stdout=False, stderr=False),  # Ray Tune's status widget
    ):
        sink = logger.add(log, format="{message}", colorize=False)
        try:
            pipeline(config_path)
        finally:
            logger.remove(sink)
            if ray.is_initialized():
                ray.shutdown()


run(design_pipeline, "configs/design.yml", RUNS / "design")
events(RUNS / "design")[["hook", "process", "model", "rows_kept"]]

,hook,process,model,rows_kept
0,__init__,this process,NaN,NaN
1,on_data_reading,this process,NaN,99769.0
2,on_dataset_creation,this process,NaN,NaN
3,on_training_complete,this process,EASE,99769.0
4,on_evaluation_complete,this process,EASE,NaN
5,on_training_complete,this process,BPR,99769.0
6,on_evaluation_complete,this process,BPR,NaN


One callback object, one process: the configuration's `__init__`, the two data
hooks, then the two model hooks for each model in the order of `models`.
`on_training_complete` still sees `rows_kept`, which `on_data_reading` set on
the same object. No Ray Tune hook was called. The table the callback wrote:

In [18]:
pd.read_csv(RUNS / "design" / "test_results.tsv", sep="\t").round(4)

,model,k,nDCG,Recall,NewReleaseShare
0,EASE,10,0.2091,0.1264,0.3189
1,BPR,10,0.1668,0.0985,0.3695


The eval and estimate pipelines also run the four WarpRec hooks in the
calling process.

## The train pipeline on Ray

The train pipeline searches hyperparameters on Ray, so the same callback runs
in several processes. The configuration adds a validation split, a `writer`
and lists of values to search, and starts a private Ray instance for the run
(`ray_address: local`, guide 10):

In [19]:
print(Path("configs/train.yml").read_text())

# The same callback and metric under the train pipeline, on a private local Ray.
# A validation split is added: the trials are scored on it.
general:
  device: cpu
  ray_address: local
  ray_verbose: 0
  custom_modules:
    - user_code/my_metrics.py
  callback:
    callback_path: user_code/my_callback.py
    callback_name: GuideCallback
    kwargs:
      items_path: ../data/ml-100k/items.tsv
      output_dir: ../runs/16-callbacks-and-custom-pipelines/train
      metric: nDCG@10
reader:
  loading_strategy: dataset
  data_type: transaction
  reading_method: local
  local_path: ../data/ml-100k/ratings.tsv
  rating_type: implicit
splitter:
  test_splitting:
    strategy: temporal_holdout
    ratio: 0.2
  validation_splitting:
    strategy: temporal_holdout
    ratio: 0.1
writer:
  dataset_name: ml-100k
  writing_method: local
  local_experiment_path: ../runs/16-callbacks-and-custom-pipelines/train
models:
  EASE:
    l2: [50.0, 200.0, 500.0]
    optimization:
      num_workers: 0
  BPR:
  

This runs three EASE trials and two four-epoch BPR trials, in under two
minutes, mostly Ray starting processes.

In [20]:
run(train_pipeline, "configs/train.yml", RUNS / "train")
train_events = events(RUNS / "train")
train_events.groupby(["hook", "process"], sort=False).size().rename("calls").to_frame()

calls
hook                   process                       
__init__               this process                 1
unpickled              data preparation worker      1
on_data_reading        data preparation worker      1
on_dataset_creation    data preparation worker      1
unpickled              this process                12
                       trial process               10
on_trial_result        this process                11
on_training_complete   this process                 2
on_evaluation_complete this process                 2

Reading the table from the top:

- The callback is built once, in this process, when the pipeline loads the
  configuration.
- It is pickled and sent to a **data preparation worker**: the train pipeline
  reads, filters and splits on Ray, so that a large dataset does not have to
  fit on the machine that drives the run. Both data hooks run there, on the
  copy.
- Back in this process, the other copies come from Ray Tune. The callback is
  referenced by the trainable WarpRec hands to Tune, so it is pickled with it:
  Tune unpickles the trainable here (12 times) while it sets up the two
  searches and schedules their trials, and **every trial's process** loads
  copies of its own (10 across the five trials).
- The Ray Tune hook, `on_training_complete` and `on_evaluation_complete` run
  in this process, on the original object.

What a copy records stays in that copy. `rows_kept`, set by `on_data_reading`
in the worker, never reaches the object here; the trial reports, collected by
the Tune hook here, do reach the model hooks:

In [21]:
train_events.loc[
    train_events.hook == "on_training_complete", ["model", "rows_kept", "trial_reports"]
]

,model,rows_kept,trial_reports
20,EASE,NaN,3.0
39,BPR,NaN,11.0


The datasets do come back, with their stash: they are what the worker
returns. Two consequences for writing a callback:

- Do not keep state in a data hook for a later hook to read; put what the
  rest of the run needs in the stash, or write it out.
- Keep the callback small. Every trial's process loads its own copy, so a
  callback holding a frame or a model would be shipped to each of them.
  `GuideCallback` holds paths and lists, and reads the items file inside the
  hooks that need it:

In [22]:
import ray.cloudpickle

print(f"{len(ray.cloudpickle.dumps(callback)):,} bytes pickled")

11,718 bytes pickled


The swarm pipeline (guide 11) moves even more onto Ray: each model's search
runs inside a Ray task of its own, so there the Tune hooks run in that task,
not in your process, and only `on_training_complete` and
`on_evaluation_complete` run where the pipeline was started.

The validation scores and training losses the Tune hook collected, one row
per trial and one column per report:

In [23]:
reports = pd.read_json(RUNS / "train" / "trial_reports.jsonl", lines=True)
reports["searched"] = [
    {k: v for k, v in p.items() if k in ("l2", "embedding_size")}
    for p in reports.params
]
reports["searched"] = reports.searched.astype(str)
reports.pivot_table(
    index=["trial", "searched"], columns="report", values=["nDCG@10", "train_loss"]
).round(4)

nDCG@10                         train_loss                        
report                                                 1       2       3       4          1       2       3       4
trial                     searched                                                                                 
BPR_DashingMistyInchworm  {'embedding_size': 16}  0.0687  0.0749  0.0864  0.0915     0.6800  0.4312  0.3074  0.2729
BPR_ElectricFierceSeriema {'embedding_size': 64}  0.0858  0.0854  0.0947  0.0918     0.6538  0.3359  0.2513  0.2111
EASE_ElatedFuriousOyster  {'l2': 50.0}            0.1174     NaN     NaN     NaN        NaN     NaN     NaN     NaN
EASE_VersedPersimmonPotoo {'l2': 500.0}           0.1184     NaN     NaN     NaN        NaN     NaN     NaN     NaN
EASE_WiseFoxOfForce       {'l2': 200.0}           0.1200     NaN     NaN     NaN        NaN     NaN     NaN     NaN

Each EASE trial reports once, with no loss: it is fitted in closed form. Each
BPR trial reports after every epoch, with that epoch's mean loss. Of the
metrics, the reports carry the validation metric only, unless
`evaluation.full_evaluation_on_report` is on: then they carry every metric of
the `evaluation` section, and the hook can record any of them.

`on_trial_result` is the Tune hook for this. `on_trial_save` is not: Tune calls
it when it saves a trial checkpoint itself, which never happens in a WarpRec
run.

The test results, collected by `on_evaluation_complete` on the best model of
each search:

In [24]:
pd.read_csv(RUNS / "train" / "test_results.tsv", sep="\t").round(4)

,model,k,nDCG,Recall,NewReleaseShare
0,EASE,10,0.1667,0.1012,0.3336
1,BPR,10,0.1382,0.0870,0.3691


EASE's search picked `l2` = 200, the design configuration's value, and it
still scores lower than in the design run: with a validation split, the main
model is trained without the validation interactions, which are each user's
most recent training rows (guide 3). The writer's own files are in
`ml-100k/` next to these (guide 10 explains them).

## A pipeline of your own

The configuration-driven pipelines cover the common protocol. When a study
needs something else (another selection rule, models trained on different
data, several datasets), the same components can be called from Python
directly. This one needs no configuration file and no Ray. It reads the
ratings, keeps the positive ones of users with at least ten, splits each
user's history in time, and builds the two datasets: train/validation for
selecting hyperparameters, train/test for the final evaluation. With a single
validation holdout both train on the same rows (guide 3), so they share their
mappings and a model built for one can be evaluated on the other.

In [25]:
from warprec.data import Dataset
from warprec.data.filtering import MinRating, UserMin, apply_filtering
from warprec.data.reader import LocalReader
from warprec.data.schema import SplitSpec
from warprec.data.splitting import Splitter

logger.disable("warprec")

data = LocalReader().read_tabular(local_path=str(folder / "ratings.tsv"), sep="\t")
data = apply_filtering(data, [MinRating(min_rating=4), UserMin(min_interactions=10)])
train, validation, test = Splitter().split_transaction(
    data,
    test=SplitSpec(strategy="temporal_holdout", ratio=0.2),
    validation=SplitSpec(strategy="temporal_holdout", ratio=0.1),
)
main = Dataset(train, test, rating_type="implicit")
val = Dataset(train, validation, rating_type="implicit", evaluation_set="Validation")

print(f"{len(data):,} rows: train {len(train):,}, validation {len(validation):,}, "
      f"test {len(test):,}")  # fmt: skip
print(
    "users, items:",
    main.get_dims(),
    "| same mappings:",
    main.get_mappings() == val.get_mappings(),
)

55,049 rows: train 38,921, validation 4,717, test 11,175
users, items: (897, 1307) | same mappings: True


### Selecting closed-form models on validation

A closed-form model is fitted when it is built, so a grid search is a loop.
Each candidate is scored on the validation dataset, by an evaluator built on
that dataset's training matrix:

In [26]:
def validation_ndcg(model) -> float:
    """nDCG@10 of a model on the validation dataset."""
    evaluator = Evaluator(["nDCG"], [10], train_set=val.train_set.get_sparse())
    evaluator.evaluate(model, val.get_evaluation_dataloader(), "full", val)
    return evaluator.compute_results()[10]["nDCG"].nanmean().item()


grid = {
    "EASE": [{"l2": l2} for l2 in (10.0, 50.0, 200.0, 500.0)],
    "ItemKNN": [{"k": k, "similarity": "cosine"} for k in (20, 50, 100)],
}
selection = pd.DataFrame(
    [
        {
            "model": name,
            "params": params,
            "nDCG@10": validation_ndcg(
                model_registry.get(
                    name, params=params, info=val.info(), interactions=val.train_set
                )
            ),
        }
        for name, candidates in grid.items()
        for params in candidates
    ]
)
best = selection.loc[selection.groupby("model")["nDCG@10"].idxmax()]
selection.round(4)

,model,params,nDCG@10
0,EASE,{'l2': 10.0},0.1129
1,EASE,{'l2': 50.0},0.1201
2,EASE,{'l2': 200.0},0.1196
3,EASE,{'l2': 500.0},0.1188
4,ItemKNN,"{'k': 20, 'similarity': 'cosine'}",0.1195
5,ItemKNN,"{'k': 50, 'similarity': 'cosine'}",0.1162
6,ItemKNN,"{'k': 100, 'similarity': 'cosine'}",0.1191


EASE with `l2` = 50 and ItemKNN with `k` = 20 score best and are kept.

### Training an iterative model as the design pipeline does

The design pipeline trains an iterative model with a `lightning.Trainer` and
`WarpRecLightningIntegrationCallback`, a Lightning callback that runs a WarpRec
`Evaluator` at the end of each validation epoch, logs its scores to Lightning,
and applies early stopping on the validation metric. Here it evaluates on the
validation dataset, which is the evaluation loader passed to `fit`. A
second, three-line Lightning callback keeps the score of each epoch. The
`Trainer` arguments are the design pipeline's, with Lightning's own logging
and progress bar switched off.

In [27]:
from warprec.recommenders.callbacks import WarpRecLightningIntegrationCallback
from warprec.utils.config.model_configuration import EarlyStopping


class EpochScores(L.Callback):
    """Keep the validation nDCG@10 of every epoch."""

    def __init__(self):
        self.scores = []

    def on_validation_epoch_end(self, trainer, pl_module):
        self.scores.append(trainer.callback_metrics["nDCG@10"].item())


def train_bpr(dataset: Dataset, epochs: int, evaluate_on: Dataset | None = None):
    """Build BPR with a fixed seed and train it, validating every epoch if asked."""
    model = model_registry.get(
        "BPR",
        params={
            "embedding_size": 64,
            "reg_weight": 1e-4,
            "batch_size": 1024,
            "epochs": epochs,
            "learning_rate": 0.005,
        },
        info=dataset.info(),
        seed=42,
    )
    callbacks, val_loader = [], None
    if evaluate_on is not None:
        callbacks = [
            WarpRecLightningIntegrationCallback(
                evaluator=Evaluator(
                    ["nDCG"], [10], train_set=evaluate_on.train_set.get_sparse()
                ),
                dataset=evaluate_on,
                strategy="full",
                early_stopping_config=EarlyStopping(patience=3, grace_period=3),
                validation_score="nDCG@10",
                mode="max",
            ),
            EpochScores(),
        ]
        val_loader = evaluate_on.get_evaluation_dataloader()
    trainer = L.Trainer(
        max_epochs=model.epochs,
        accelerator="cpu",
        num_sanity_val_steps=0,
        logger=False,
        enable_checkpointing=False,
        enable_progress_bar=False,
        enable_model_summary=False,
        check_val_every_n_epoch=1,
        callbacks=callbacks,
    )
    trainer.fit(
        model,
        train_dataloaders=model.get_dataloader(
            interactions=dataset.train_set, sessions=dataset.train_session
        ),
        val_dataloaders=val_loader,
    )
    return model, (callbacks[-1].scores if callbacks else None)


_, epoch_scores = train_bpr(val, epochs=30, evaluate_on=val)
best_epoch = 1 + max(range(len(epoch_scores)), key=epoch_scores.__getitem__)
print("validation nDCG@10 per epoch:", [round(s, 4) for s in epoch_scores])
print("best epoch:", best_epoch)

validation nDCG@10 per epoch: [0.0342, 0.0799, 0.0806, 0.0815, 0.0874, 0.0941, 0.0941, 0.0984, 0.0978, 0.097, 0.0969]
best epoch: 8


The score peaks at epoch 8, and early stopping ends the run after three
epochs without improvement. The model left at that point is the last epoch's,
not the best one. The train pipeline handles this by retraining the best
configuration for its best number of epochs; the same here, on the main
dataset. With the same seed and the same training rows, this reproduces the
model of epoch 8, as its validation score shows:

In [28]:
bpr, _ = train_bpr(main, epochs=best_epoch)
print(f"validation nDCG@10 of the retrained model: {validation_ndcg(bpr):.4f}")

validation nDCG@10 of the retrained model: 0.0984


### Testing, comparing and writing

The selected models are tested once, on the main dataset, by one evaluator:

In [29]:
final = {
    row.model: model_registry.get(
        row.model, params=row.params, info=main.info(), interactions=main.train_set
    )
    for row in best.itertuples()
}
final["BPR"] = bpr

tester = Evaluator(
    ["nDCG", "Recall", "Precision"], [10, 20], train_set=main.train_set.get_sparse()
)
test_results = {}
for name, model in final.items():
    tester.evaluate(model, main.get_evaluation_dataloader(), "full", main)
    test_results[name] = tester.compute_results()

pd.DataFrame(
    {
        name: {
            f"{metric}@{k}": value.nanmean().item()
            for k, metrics in results.items()
            for metric, value in metrics.items()
        }
        for name, results in test_results.items()
    }
).T.round(4)

,nDCG@10,Recall@10,Precision@10,nDCG@20,Recall@20,Precision@20
EASE,0.1311,0.1141,0.1002,0.1545,0.1930,0.0871
ItemKNN,0.1298,0.1142,0.0968,0.1493,0.1826,0.0812
BPR,0.1134,0.0964,0.0892,0.1332,0.1647,0.0780


A paired Wilcoxon test on every pair of models, metric and cutoff, with the
Holm–Bonferroni correction over all of them (guide 12 covers the tests). The
cutoff-10 rows:

In [30]:
from warprec.evaluation import compute_paired_statistical_test

significance = compute_paired_statistical_test(
    test_results, "wilcoxon_test", holm_bonferroni=True, backend="pandas"
)
table = significance.to_native()
table.columns = [c.split(" (")[-1].rstrip(")") for c in table.columns]
table[table.Cutoff == 10].drop(columns="Statistic").set_index(
    ["Metric", "Model A", "Model B"]
).round(6)

Cutoff   p-value           α=0.05  Holm-Bonferroni
Metric    Model A Model B                                                    
nDCG      EASE    BPR          10  0.000176      Significant      Significant
Recall    ItemKNN BPR          10  0.000193      Significant      Significant
          EASE    BPR          10  0.000221      Significant      Significant
nDCG      ItemKNN BPR          10  0.000252      Significant      Significant
Precision EASE    BPR          10  0.001069      Significant      Significant
          ItemKNN BPR          10  0.018118      Significant  Not significant
          EASE    ItemKNN      10  0.236706  Not significant  Not significant
nDCG      EASE    ItemKNN      10  0.291167  Not significant  Not significant
Recall    EASE    ItemKNN      10  0.803441  Not significant  Not significant

At cutoff 10, BPR is worse than both closed-form models on every metric,
significantly so after the correction except for Precision against ItemKNN;
EASE and ItemKNN cannot be told apart.

A `LocalWriter` writes the same files the train pipeline writes: one results
table for all models, the parameters, the test table, and here the top-10 of
the best model. Its `timestamp` is fixed so that the file names are stable.

In [31]:
from warprec.data.writer import LocalWriter

writer = LocalWriter(
    dataset_name="ml-100k", local_path=str(RUNS / "custom"), timestamp="guide"
)
for name, results in test_results.items():
    writer.write_results(results, name)
writer.write_params(
    {row.model: row.params for row in best.itertuples()}
    | {"BPR": {**bpr.get_params(), "epochs": best_epoch}}
)
writer.write_statistical_significance_test(significance, "wilcoxon_test")
with contextlib.redirect_stderr(io.StringIO()):  # a progress bar
    writer.write_recs(final["EASE"], main, k=10)

sorted(
    str(p.relative_to(writer.experiment_path))
    for p in writer.experiment_path.rglob("*")
    if p.is_file()
)

['evaluation/Overall_Results_guide.tsv',
 'evaluation/Wilcoxon_test_guide.tsv',
 'params/Overall_Params_guide.json',
 'recs/EASE_guide.tsv']

In [32]:
pd.read_csv(
    writer.experiment_evaluation_path / "Overall_Results_guide.tsv", sep="\t"
).round(4)

,Model,Top@k,Precision,Recall,nDCG
0,BPR,10,0.0892,0.0964,0.1134
1,BPR,20,0.0780,0.1647,0.1332
2,EASE,10,0.1002,0.1141,0.1311
3,EASE,20,0.0871,0.1930,0.1545
4,ItemKNN,10,0.0968,0.1142,0.1298
5,ItemKNN,20,0.0812,0.1826,0.1493


Compared with the train pipeline, this pipeline gives up Ray (parallel trials
on a cluster, schedulers, resuming), the time report, model serialization
and the configuration's validation, and gains complete control of the
protocol. Anything in between can be done with a callback.

## Summary

| what | where |
|---|---|
| `general.callback.callback_path` / `callback_name` | the `.py` file (relative to the directory WarpRec is started from) and the `WarpRecCallback` subclass in it; checked when the configuration loads |
| `general.callback.args` / `kwargs` | passed unvalidated to the constructor |
| `on_data_reading(data)` | after the reader, before `filtering` and the split, with `loading_strategy: dataset` only; returns the frame |
| `on_dataset_creation(main_dataset, val_dataset, validation_folds)` | once all datasets exist; fill the stash of each |
| `on_training_complete(model)`, `on_evaluation_complete(model, params, results)` | once per model, after training and after the test evaluation |
| Ray Tune hooks (`on_trial_result`, ...) | train and swarm pipelines only |
| where hooks run | design, eval, estimate: the calling process. train: data hooks in a Ray worker, the rest in the calling process. swarm: data hooks in a Ray worker, Tune hooks in each model's Ray task, model hooks in the calling process |
| callback state | a copy per process: state set in a hook stays in the process that ran it; keep the object small, every trial loads it |
| stash | `add_to_stash` / `get_stash`; unpacked into every model's and every metric's constructor |

[Callbacks](https://warprec.readthedocs.io/en/latest/extending/callbacks/)
describes the callback interface,
[Stash](https://warprec.readthedocs.io/en/latest/extending/stash/) the stash,
and [General configuration](https://warprec.readthedocs.io/en/latest/configuration/general/)
the `callback` section. Guide 15 covers custom models and metrics, guide 17
serving.